# Hybrid Quantum-Classical Siamese Network
Production-ready architecture for fingerprint verification.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install qiskit qiskit-machine-learning qiskit-aer --upgrade -q
!pip install pylatexenc matplotlib opencv-python faiss-cpu -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import os
import copy

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("Seeds set. Device:", torch.device("cpu"))

In [ ]:
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def get_label(filename):
    return filename.split("__")[0]

def apply_clahe(img):
    img_np = np.array(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(img_np)
    return Image.fromarray(img_clahe)

train_transform = transforms.Compose([
    transforms.Lambda(apply_clahe),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transform = transforms.Compose([
    transforms.Lambda(apply_clahe),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

def build_pairs(image_paths, num_pos_pairs=5):
    label_to_images = {}
    for f in image_paths:
        label = get_label(f)
        label_to_images.setdefault(label, []).append(f)
    
    # Skip subjects with only one image
    labels = [l for l, imgs in label_to_images.items() if len(imgs) >= 2]
    pairs = []
    
    for label in labels:
        imgs = label_to_images[label]
        # Limit to the max unique combinations possible
        max_pos = len(imgs) * (len(imgs) - 1) // 2
        actual_pairs = min(num_pos_pairs, max_pos)
        
        for _ in range(actual_pairs):
            img1, img2 = random.sample(imgs, 2)
            pairs.append((img1, img2, 1))
            
    num_pos = len(pairs)
    for _ in range(num_pos):
        l1, l2 = random.sample(labels, 2)
        img1 = random.choice(label_to_images[l1])
        img2 = random.choice(label_to_images[l2])
        pairs.append((img1, img2, 0))
        
    random.shuffle(pairs)
    return pairs

class SiameseFingerprintDataset(Dataset):
    def __init__(self, root_dir, image_list, transform=None, num_pos_pairs=5):
        self.root_dir = root_dir
        self.transform = transform
        self.image_list = image_list
        self.pairs = build_pairs(image_list, num_pos_pairs=num_pos_pairs)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_name, img2_name, label = self.pairs[idx]
        img1 = Image.open(os.path.join(self.root_dir, img1_name)).convert("L")
        img2 = Image.open(os.path.join(self.root_dir, img2_name)).convert("L")
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.float32)

dataset_path = '/content/drive/MyDrive/SOCOFing/Real'
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset path not found: {dataset_path}. Check your Drive mount.")
else:
    all_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(".bmp")]

# Subject-disjoint split
unique_labels = list(set([get_label(f) for f in all_files]))
random.shuffle(unique_labels)

n_train = int(0.7 * len(unique_labels))
n_val = int(0.15 * len(unique_labels))

train_labels = set(unique_labels[:n_train])
val_labels = set(unique_labels[n_train:n_train+n_val])
test_labels = set(unique_labels[n_train+n_val:])

train_files = [f for f in all_files if get_label(f) in train_labels]
val_files = [f for f in all_files if get_label(f) in val_labels]
test_files = [f for f in all_files if get_label(f) in test_labels]

set_seed(42) # Lock pairs
train_dataset = SiameseFingerprintDataset(dataset_path, train_files, transform=train_transform, num_pos_pairs=2)
val_dataset = SiameseFingerprintDataset(dataset_path, val_files, transform=val_transform, num_pos_pairs=2)
test_dataset = SiameseFingerprintDataset(dataset_path, test_files, transform=val_transform, num_pos_pairs=2)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train pairs: {len(train_dataset)} | Val pairs: {len(val_dataset)} | Test pairs: {len(test_dataset)}")

In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

device = torch.device("cpu")

resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)

for name, param in resnet.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
original_conv1 = models.resnet18(weights=ResNet18_Weights.DEFAULT).conv1.weight.data
new_conv1 = original_conv1.mean(dim=1, keepdim=True)
resnet.conv1.weight.data = new_conv1

resnet.fc = nn.Linear(512, 4)
resnet = resnet.to(device)
print("ResNet18 ready")

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

num_qubits = 4
qc = QuantumCircuit(num_qubits)
input_params = [Parameter(f"x{i}") for i in range(4)]
weights = [Parameter(f"w{i}") for i in range(4)]

# Ultra-lightweight Architecture to bypass CPU bottleneck
for i in range(4):
    qc.rx(input_params[i], i)
    
qc.cx(0,1)
qc.cx(1,2)
qc.cx(2,3)
qc.cx(3,0)

for i in range(4):
    qc.ry(weights[i], i)

observable = SparsePauliOp.from_list([
    ("ZIII", 1), ("IZII", 1), ("IIZI", 1), ("IIIZ", 1),
    ("ZZII", 0.5), ("IIZZ", 0.5)
])

qnn = EstimatorQNN(
    circuit=qc,
    estimator=AerEstimator(),
    observables=observable,
    input_params=input_params,
    weight_params=weights,
    input_gradients=True
)

quantum_layer = TorchConnector(qnn)
print("Quantum layer ready")

In [ ]:
class HybridQNNModel(nn.Module):
    def __init__(self, base_model, q_layer):
        super().__init__()
        self.base = base_model
        self.qnn = q_layer
        self.scale = nn.Tanh()

    def forward(self, x1, x2):
        f1 = self.base(x1)
        f2 = self.base(x2)
        diff = torch.abs(f1 - f2) # Absolute difference for symmetry
        diff_scaled = self.scale(diff) * torch.pi
        raw = self.qnn(diff_scaled)
        sim = (raw + 5) / 10 # Normalize to [0,1] since max bound shifted upwards to 5 due to ZZ observables
        return sim.squeeze()

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
    def forward(self, sim, label):
        loss_pos = label * (1 - sim) ** 2
        loss_neg = (1 - label) * torch.clamp(sim - self.margin, min=0) ** 2
        return torch.mean(loss_pos + loss_neg)

criterion = ContrastiveLoss(margin=0.3)

In [ ]:
from torch.nn.utils import clip_grad_norm_
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

q_model = HybridQNNModel(copy.deepcopy(resnet), quantum_layer).to(device)

optimizer_q = torch.optim.Adam([
    {'params': q_model.qnn.parameters(), 'lr': 0.001},
    {'params': q_model.base.parameters(), 'lr': 0.0005}
], weight_decay=1e-5)

epochs = 5
patience = 2
best_val_loss_q = float('inf')
wait = 0
train_losses_q = []
val_losses_q = []

for epoch in range(epochs):
    q_model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} (Quantum Train)")
    for img1, img2, label in pbar:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        optimizer_q.zero_grad()
        sim = q_model(img1, img2)
        loss = criterion(sim, label)
        loss.backward()
        clip_grad_norm_(q_model.parameters(), max_norm=1.0)
        optimizer_q.step()
        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_train_loss / len(train_loader)
    train_losses_q.append(avg_train_loss)

    q_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for img1, img2, label in val_loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            sim = q_model(img1, img2)
            loss = criterion(sim, label)
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_losses_q.append(avg_val_loss)
    print(f"Quantum Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss_q:
        best_val_loss_q = avg_val_loss
        wait = 0
        torch.save(q_model.state_dict(), '/content/drive/MyDrive/SOCOFing/hybrid_qnn_best.pth')
        print("   >> Saved best model")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

plt.figure(figsize=(8,4))
plt.plot(train_losses_q, label='Quantum Train Loss')
plt.plot(val_losses_q, label='Quantum Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Quantum Model Training')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d

q_model.load_state_dict(torch.load('/content/drive/MyDrive/SOCOFing/hybrid_qnn_best.pth', map_location=device))
q_model.eval()
q_scores = []
q_labels = []

with torch.no_grad():
    for img1, img2, label in test_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        sim = q_model(img1, img2)
        q_scores.extend(sim.cpu().numpy().flatten())
        q_labels.extend(label.cpu().numpy().flatten())

fpr_q, tpr_q, thresholds_q = roc_curve(q_labels, q_scores)
roc_auc_q = auc(fpr_q, tpr_q)
eer_q = brentq(lambda x: 1. - x - interp1d(fpr_q, tpr_q)(x), 0., 1.)
thresh_q = interp1d(fpr_q, thresholds_q)(eer_q)
preds_q = (np.array(q_scores) >= thresh_q).astype(int)

print(f"\n=== Quantum Evaluation ===")
print(f"AUC: {roc_auc_q:.4f} | EER: {eer_q:.4f} (Thresh: {thresh_q:.3f})")
print(f"Accuracy:  {accuracy_score(q_labels, preds_q)*100:.2f}%")
print(f"Precision: {precision_score(q_labels, preds_q, zero_division=0)*100:.2f}%")
print(f"Recall:    {recall_score(q_labels, preds_q, zero_division=0)*100:.2f}%")
print(f"F1-Score:  {f1_score(q_labels, preds_q, zero_division=0)*100:.2f}%")

In [ ]:
class ClassicalSiamese(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.classical_head = nn.Sequential(
            nn.Linear(8, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x1, x2):
        f1 = self.base(x1)
        f2 = self.base(x2)
        diff = torch.abs(f1 - f2)
        multi = f1 * f2
        combined = torch.cat((diff, multi), dim=1)
        return self.classical_head(combined).squeeze()

c_model = ClassicalSiamese(copy.deepcopy(resnet)).to(device)

# Unfreeze deeper layers explicitly on the classical copy (no effect to quantum)
for param in c_model.base.layer2.parameters():
    param.requires_grad = True
for param in c_model.base.layer3.parameters():
    param.requires_grad = True

criterion_c = nn.BCEWithLogitsLoss()

optimizer_c = torch.optim.Adam([
    {'params': c_model.classical_head.parameters(), 'lr': 0.001},
    {'params': filter(lambda p: p.requires_grad, c_model.base.parameters()), 'lr': 0.0001}
], weight_decay=1e-5)

epochs = 5
patience = 2
best_val_loss_c = float('inf')
wait = 0
train_losses_c = []
val_losses_c = []

for epoch in range(epochs):
    c_model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} (Classic Train)")
    for img1, img2, label in pbar:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        optimizer_c.zero_grad()
        sim = c_model(img1, img2)
        loss = criterion_c(sim, label)
        loss.backward()
        clip_grad_norm_(c_model.parameters(), max_norm=1.0)
        optimizer_c.step()
        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_train_loss / len(train_loader)
    train_losses_c.append(avg_train_loss)

    c_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for img1, img2, label in val_loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            sim = c_model(img1, img2)
            loss = criterion_c(sim, label)
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_losses_c.append(avg_val_loss)
    print(f"Classic Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss_c:
        best_val_loss_c = avg_val_loss
        wait = 0
        torch.save(c_model.state_dict(), '/content/drive/MyDrive/SOCOFing/classic_siamese_best.pth')
        print("   >> Saved best model")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

plt.figure(figsize=(8,4))
plt.plot(train_losses_c, label='Classic Train Loss')
plt.plot(val_losses_c, label='Classic Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Classical Model Training')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
c_model.load_state_dict(torch.load('/content/drive/MyDrive/SOCOFing/classic_siamese_best.pth', map_location=device))
c_model.eval()
c_scores = []
c_labels = []

with torch.no_grad():
    for img1, img2, label in test_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        sim = c_model(img1, img2)
        c_scores.extend(torch.sigmoid(sim).cpu().numpy().flatten())
        c_labels.extend(label.cpu().numpy().flatten())

fpr_c, tpr_c, thresholds_c = roc_curve(c_labels, c_scores)
roc_auc_c = auc(fpr_c, tpr_c)
eer_c = brentq(lambda x: 1. - x - interp1d(fpr_c, tpr_c)(x), 0., 1.)
thresh_c = interp1d(fpr_c, thresholds_c)(eer_c)
preds_c = (np.array(c_scores) >= thresh_c).astype(int)

print(f"\n=== Classical Evaluation ===")
print(f"AUC: {roc_auc_c:.4f} | EER: {eer_c:.4f} (Thresh: {thresh_c:.3f})")
print(f"Accuracy:  {accuracy_score(c_labels, preds_c)*100:.2f}%")
print(f"Precision: {precision_score(c_labels, preds_c, zero_division=0)*100:.2f}%")
print(f"Recall:    {recall_score(c_labels, preds_c, zero_division=0)*100:.2f}%")
print(f"F1-Score:  {f1_score(c_labels, preds_c, zero_division=0)*100:.2f}%")

plt.figure(figsize=(8,6))
plt.plot(fpr_q, tpr_q, label=f'Quantum ROC (AUC = {roc_auc_q:.3f})', color='orange')
plt.plot(fpr_c, tpr_c, label=f'Classical ROC (AUC = {roc_auc_c:.3f})', color='blue')
plt.plot([0,1], [0,1], 'k--')
plt.scatter(eer_q, 1-eer_q, marker='D', color='red', label=f'Quantum EER = {eer_q:.3f}')
plt.scatter(eer_c, 1-eer_c, marker='D', color='cyan', label=f'Classical EER = {eer_c:.3f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Quantum vs Classical ROC Comparison')
plt.legend()
plt.grid()
plt.show()